In [1]:
from __future__ import print_function, division

%matplotlib inline

import os
import pylab as pl
import matplotlib.pyplot as plt
from glob import glob

import numpy as np
import tensorflow as tf
import pandas as pd
import pickle as pkl
import random as python_random

tf.autograph.set_verbosity(0)

from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.metrics import top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

from tensorflow.keras.layers import Input, Dense, Layer
from tensorflow.keras.losses import BinaryCrossentropy, CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from IPython import display
from typing import List, Tuple

# reproducible
np.random.seed(123)
python_random.seed(123)
tf.random.set_seed(1234)

In [2]:
from malware_detection_inference import MalwareDetection

In [3]:
# In progress...

class AugmentMalwareImages:
    def __init__(image_data_dir : str, balance_ratio : Tuple):
        self.image_data_dir = image_data_dir
        self.balance_ratio = balance_ratio
        
    

In [3]:
class MalwareImageGAN(object):
    def __init__(self, source_images_dir, target_images_dir, 
                 source_input_shape : Tuple, target_input_shape : Tuple,# for now both source and target shape must be same
                 conv_model_path : str, nSteps = 200): 
        """
        Argument:
            source_images_dir : directory for source images (train and test)
            target_train_dir : target images for training
            nSteps : num of iterations|epochs for train the model

        """
        # Source and Target directories
        self.source_images_dir = source_images_dir
        self.target_images_dir = target_images_dir
        
        # Source train and test dataset
        self.source_train_images = self.source_images_dir + 'train/'
        self.source_test_images = self.source_images_dir + 'test/'
    
        # Target train and test dataset
        
        self.target_train_images = self.target_images_dir + 'train/'        
        self.target_test_images = self.target_images_dir + 'test/'
        
        self.source_input_shape = source_input_shape
        self.target_input_shape = target_input_shape
        
        self.conv_model_path = conv_model_path
        
#         self.n_classes = y_source_train.shape[1]   # need to discuss about this line of code
        self.n_classes = 2
                        
        # Use the source dataset shape for the generator input and outputs.
        self.input_shape_S = source_input_shape     # (320x320)
        self.input_shape_T = target_input_shape     # (320x320)
        
        #Latent dim for AE/VAE

        self.latent_dim = 256
        
        self.optimizer = Adam(0.0002, 0.5)  # Adam(1e-5)
        self.batch_size = 2
        self.nStep = nSteps

        # folder contains performance figures written-out periodically
        if not os.path.exists("./fig/"):
            os.makedirs("./fig/")

    def conv_model(self):
        malwareConvNet = MalwareDetection(model_path = self.conv_model_path,
                          optimizer = Adam(lr = 0.001),
                          loss_fn = 'sparse_categorical_crossentropy',
                          metrics = ['accuracy'],
                          input_shape = self.source_input_shape)
        return malwareConvNet.model
    
    def build_generator_S(self):
        print("\n== Build Generator S...")
        
        model = self.conv_model()
        G_S = Model(inputs = model.inputs, outputs = [model.layers[-2].output], name = "Generator_S")
        return G_S

      
    def build_generator_T(self):
        print("\n== Build Generator T...")
        
        model = self.conv_model()
        G_T = Model(inputs = model.inputs, outputs = [model.layers[-2].output], name = "Generator_T")
        return G_T

    def build_generator(self):
        print("\n== Build Generator...")
        
        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G2")(net)        
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G4")(net)

        DIrep = Dense(units = self.latent_dim, activation = tf.nn.sigmoid, name = "DIrep")(net)
        G = Model(inputs = inputs, outputs = DIrep, name = "Generator")
        
        #Classifier
        inputs = Input(DIrep.shape)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C2")(net)
        net = Dense(units = self.n_classes, activation = tf.nn.softmax, name = "C")(net)
        C = Model(inputs = inputs, outputs = net, name = "Classifier")
        
        return G, C

    def build_disciminator(self):
        print("\n== Build Discriminator...")
        
        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D2")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D4")(net)

        net = Dense(units = 2, activation = tf.nn.softmax, name = "D")(net)
        D = Model(inputs = inputs, outputs = net, name = "Discriminator")
        return D


    def d_loss(self, yhat_source, yhat_target):        
        y_source = np.tile([1,0], (yhat_source.shape[0], 1))
        y_target = np.tile([0,1], (yhat_target.shape[0], 1))
                          
        bce = CategoricalCrossentropy(from_logits = False)        
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def g_loss(self, yhat_source, yhat_target):
        #[0,1]
        #[0,1]
        # ...
        
        y_source = np.tile([0,1], (yhat_source.shape[0], 1))
        y_target = np.tile([1,0], (yhat_target.shape[0], 1))
                          
        bce = CategoricalCrossentropy(from_logits = False)        
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def c_loss(self, yhat_class_source, yhat_class_target, y_source, y_target):
        #source_weight = 0.5
        #target_weight = 1 
        bce = CategoricalCrossentropy(from_logits = False)
#         print(y_source.dtype)
#         print(yhat_class_source.dtype)
        #return (source_weight*bce(y_source, yhat_class_source) + target_weight* bce(y_target, yhat_class_target))/(source_weight + target_weight)
        return bce(y_source, yhat_class_source) + bce(y_target, yhat_class_target) #weight-source. bce() + .../(ws+wtt)
        
    def create_image_tensor(self):
        """
            This method is used to create image tensors for testing folder
        """
        
        S_test_images = []
        S_test_labels = []
        T_test_images = []
        T_test_labels = []
        for im in glob(self.source_test_images + '/**/*'):
            label = 0 if im.startswith('benign') else 1     # images name must start with benign_(someinteger).png
                                                            # and malicious_(someinteger).png
            img = tf.io.read_file(im)
            tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
            tensor = tf.image.resize(tensor, list(self.input_shape_S))
            # input_tensor = tf.expand_dims(tensor, axis = 0)
            S_test_images.append(tensor)
            S_test_labels.append(label)
        
        for im in glob(self.target_test_images + '/**/*'):
            label = 0 if im.startswith('benign') else 1     # images name must start with benign_(someinteger).png
                                                            # and malicious_(someinteger).png
            img = tf.io.read_file(im)
            tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
            tensor = tf.image.resize(tensor, list(self.input_shape_T))
            # input_tensor = tf.expand_dims(tensor, axis = 0)
            T_test_images.append(tensor)
            T_test_labels.append(label)
        
        S_test_images = tf.convert_to_tensor(S_test_images)
        S_test_labels = tf.convert_to_tensor(S_test_labels)
        T_test_images = tf.convert_to_tensor(T_test_images)
        T_test_labels = tf.convert_to_tensor(T_test_labels)
        return S_test_images, S_test_labels, T_test_images, T_test_labels
    
    
    def train(self):
        D = self.build_disciminator()
        G_S = self.build_generator_S()
        G_T = self.build_generator_T()
        G, C = self.build_generator()
        
        S_batches = tf.keras.preprocessing.image_dataset_from_directory(self.source_train_images,
                                                      seed = 123,
                                                      image_size = self.source_input_shape,
                                                      batch_size = self.batch_size)
        
        T_batches = tf.keras.preprocessing.image_dataset_from_directory(self.target_train_images,
                                                      seed = 123,
                                                      image_size = self.target_input_shape,
                                                      batch_size = self.batch_size)
        
        S_test_images, S_test_labels, T_test_images, T_test_labels = self.create_image_tensor()
        
        # NOTE -> without batch size maximum no. of images must be 32 in testing folders
        
        S_batches = iter(S_batches)
        T_batches = iter(T_batches)
        
        optimizer = self.optimizer

        g_loss_weight = 1
        c_loss_weight = 1
       

        print('====Loss Weights====')
        print('g_loss_weight: {0}'.format(g_loss_weight))
        print('c_loss_weight: {0}'.format(c_loss_weight))
       

        #@tf.function
        def _train_step():

            # Get a batch of source and target unlabeled samples
            x_batch_source, y_batch_source = next(S_batches)
            x_batch_target, y_batch_target = next(T_batches)
   
            #Create feature selections 
            feature_S = G_S(x_batch_source)
            feature_T = G_T(x_batch_target)    
    
            #Create domain invariant mapping using the Generator
            DIrep_source_samples = G(feature_S)
            DIrep_target_samples = G(feature_T)

            #if concatenate, there will be only one DIrep 
            #what changes happen to d_loss g_loss c_loss


            # Calculate the Domain loss
            with tf.GradientTape(persistent = True) as tape_disc:
                
                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target = D(DIrep_target_samples)
                
                # Compute D loss
                d_loss_value = self.d_loss(yhat_source, yhat_target)
                
            # Given loss, compute and apply gradient for discriminator:
            d_gradients = tape_disc.gradient(d_loss_value, D.trainable_variables)
            optimizer.apply_gradients(zip(d_gradients, D.trainable_variables))
            
           
            # Get a batch of source and target unlabeled samples
            x_batch_source, y_batch_source = next(S_batches)
            x_batch_target, y_batch_target = next(T_batches)
            
       
            with tf.GradientTape(persistent = True) as tape_gen:
                
                #Create feature selections 
                feature_S = G_S(x_batch_source)
                feature_T = G_T(x_batch_target) 

                #Create domain invariant mapping using the Generator
                DIrep_source_samples = G(feature_S)
                DIrep_target_samples = G(feature_T)


                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target= D(DIrep_target_samples)
                

                #Predict the class of the samples
                class_pred_source = tf.math.argmax(C(DIrep_source_samples))
                
                class_pred_target = tf.math.argmax(C(DIrep_target_samples))
                
            
                # Compute G loss
                g_loss_value = self.g_loss(yhat_source, yhat_target)
                
                # Compute C loss

                c_loss_value = self.c_loss(tf.cast(class_pred_source, tf.float32), tf.cast(class_pred_target, tf.float32), 
                                      tf.cast(y_batch_source, tf.float32), tf.cast(y_batch_target, tf.float32))
                
                combined_loss_value = (g_loss_weight * g_loss_value + c_loss_weight * c_loss_value) / (g_loss_weight + c_loss_weight)

                
                
            # Given loss, compute and apply gradient:
#             print(combined_loss_value.dtype)
#             print(c_loss_value.dtype)
            
#             print(combined_loss_value)
#             print(c_loss_value)
            
            c_gradients = tape_gen.gradient(c_loss_value, C.trainable_variables)
            gs_gradients = tape_gen.gradient(combined_loss_value, G_S.trainable_variables)
            gt_gradients = tape_gen.gradient(combined_loss_value, G_T.trainable_variables)
            g_gradients = tape_gen.gradient(combined_loss_value, G.trainable_variables)

#             print(g_gradients)
#             print(c_gradients)
            
            optimizer.apply_gradients(zip(gs_gradients, G_S.trainable_variables))
            optimizer.apply_gradients(zip(gt_gradients, G_T.trainable_variables))
            optimizer.apply_gradients(zip(g_gradients, G.trainable_variables))
            optimizer.apply_gradients(zip(c_gradients, C.trainable_variables))
            
            
            return G_S, G_T, G, C, D, g_loss_value, c_loss_value, d_loss_value, combined_loss_value

        # Start training nStep:
        #fig, ax = plt.subplots()
        for step in range(self.nStep):
            generator_S, generator_T, generator, classifier, discriminator, g_loss_value, c_loss_value, d_loss_value, combined_loss_value = _train_step()
            
            if step % 10 == 0:
                
                y_source_pred_test = classifier.predict(generator(generator_S(S_test_images))).argmax(1)
                y_target_pred_test = classifier.predict(generator(generator_T(T_test_images))).argmax(1)
                
                
                accuracy_source = accuracy_score(S_test_labels, y_source_pred_test)
                accuracy_target = accuracy_score(T_test_labels, y_target_pred_test)

                y_source_DI_test = generator(generator_S(S_test_images))
                y_target_DI_test = generator(generator_T(T_test_images))
                

                y_source_domain_pred = discriminator(y_source_DI_test).numpy().argmax(1)
                y_target_domain_pred = discriminator(y_target_DI_test).numpy().argmax(1)
                #y_domain_pred = tf.concat([y_source_domain_pred, y_target_domain_pred], axis=0)
                #why it is 1,0?
                y_domain_source_real = np.array([1] * y_source_domain_pred.shape[0])
                y_domain_target_real = np.array([0] * y_target_domain_pred.shape[0])
                #y_domain_real =  tf.concat([y_domain_source_real, y_domain_target_real], axis=0)
                
                #print((y_domain_pred.numpy() == 1).sum())
                #print((y_domain_real.numpy() == 1).sum())
                
                domain_pred_accuracy_source = accuracy_score(y_domain_source_real, y_source_domain_pred)
                domain_pred_accuracy_target = accuracy_score(y_domain_target_real, y_target_domain_pred)

                f1_source = f1_score(S_test_labels, y_source_pred_test, average = 'weighted')
                f1_target = f1_score(T_test_labels, y_target_pred_test, average = 'weighted')

                
                track_loss = '\nStep %4d ==>Comb_loss: %4.4f G_Loss: %4.4f C_Loss: %4.4f D_Loss: %4.4f \n Acc Source: %4.4f Acc Target: %4.4f F1 Source: %4.4f F1 Target: %4.4f \n Acc Domain Source: %4.4f  Acc Domain Target: %4.4f' % (
                                                                                                    step, combined_loss_value, g_loss_value.numpy(), c_loss_value.numpy(), d_loss_value.numpy(), 
                                                                                                    accuracy_source, accuracy_target, f1_source, f1_target,
                                                                                                    domain_pred_accuracy_source, domain_pred_accuracy_target)
                print(track_loss)
                
        print('Training ended')

In [4]:
malware_gan = MalwareImageGAN(
                              source_images_dir = 'Data/source/',
                              target_images_dir = 'Data/target/',
                              source_input_shape = (320, 320),
                              target_input_shape = (320, 320), 
                              conv_model_path = 'weights/resnet_101_320x320.h5'
)

In [5]:
malware_gan.train()


== Build Discriminator...

== Build Generator S...

== Build Generator T...

== Build Generator...
Found 46 files belonging to 2 classes.
Found 44 files belonging to 2 classes.
====Loss Weights====
g_loss_weight: 1
c_loss_weight: 1

Step    0 ==>Comb_loss: 16.8305 G_Loss: 1.4247 C_Loss: 32.2362 D_Loss: 1.3981 
 Acc Source: 0.0000 Acc Target: 0.0000 F1 Source: 0.0000 F1 Target: 0.0000 
 Acc Domain Source: 1.0000  Acc Domain Target: 0.0000

Step   10 ==>Comb_loss: 16.7801 G_Loss: 1.3240 C_Loss: 32.2362 D_Loss: 1.4238 
 Acc Source: 0.0000 Acc Target: 0.0000 F1 Source: 0.0000 F1 Target: 0.0000 
 Acc Domain Source: 1.0000  Acc Domain Target: 0.0000


StopIteration: 